# Timing a planner fairly

Speed claims are easy to get wrong. This notebook times the canonical A\* planner of `passage-eval` on
validation tasks and shows the practices that make a timing meaningful: compile and warm up before
measuring, repeat each request, report percentiles rather than a single mean, and state the machine.
The release metadata keeps no generation times, so every number here is measured by this notebook.

The numbers describe this run on this machine. They are not a benchmark of A\*, and they cannot be
compared with timings taken on other hardware.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It solves each task several times; about one minute at 256×256.

In [ ]:
resolution = 256
split = "validation"
objective = "elevation"
condition = "obstacles"
tasks = 32  # first validation scenes timed
repeats = 5  # timed repetitions per task, after one warm-up call
output_dir = "notebook-artifacts/notebooks/astar_inference_timing"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))

## 1. The machine and the tasks

Timings depend on the processor, the libraries and the load of the machine. Table 1 records what this
run used. Tasks are prepared in memory before any timing starts, so reading and decoding the data never
enter the measurement.

In [ ]:
import os
import platform
import time
import numba
from passage_eval.io.dataset import scene_from_row
from passage_eval.baselines.classical import AStarPlanner

machine = pd.DataFrame([
    {"Item": "Processor architecture", "Value": platform.machine()},
    {"Item": "Logical cores visible", "Value": str(len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count())},
    {"Item": "Python / NumPy / Numba", "Value": f"{platform.python_version()} / {np.__version__} / {numba.__version__}"},
    {"Item": "Threads used by the planner", "Value": "1 (sequential requests)"},
])
show(machine, "Table 1. Execution context of this run.")
SOLVER_OF = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}
tasks_list = []
for r in source.iter_rows(resolution, split, columns=None, samples=False):
    if r["metadata"]["paths"][SOLVER_OF[objective]][condition]["status"] == "ok":  # every released variant is "ok"
        tasks_list.append(scene_from_row(r, resolution=resolution, cost_model=objective, split=split,
                                         task_variant=condition))
    if len(tasks_list) == tasks:
        break

## 2. Compilation and warm-up

The grid search is compiled on its first call. That one-off cost belongs to the start-up of a service,
not to each request, so it is measured apart.

In [ ]:
planner = AStarPlanner()
began = time.perf_counter()
planner.predict_path(tasks_list[0])
first_call = time.perf_counter() - began
began = time.perf_counter()
planner.predict_path(tasks_list[0])
second_call = time.perf_counter() - began
display(Markdown(f"First call, which compiles the search or loads it from the compilation cache: **{first_call:.2f} s**. Second call on the same task: "
                 f"**{1000 * second_call:.1f} ms**."))

## 3. Warm timings

Each task gets one untimed warm-up call, then `repeats` timed calls. The per-task time is the median of
its repetitions; the spread across tasks reflects the tasks themselves.

In [ ]:
timings = []
for task in tasks_list:
    planner.predict_path(task)
    samples = []
    for _ in range(repeats):
        began = time.perf_counter()
        output = planner.predict_path(task)
        samples.append(time.perf_counter() - began)
    timings.append({"Scene": task.sample_id, "Route cells": len(output.predicted_path),
                    "Obstacle coverage (%)": 100 * float(np.mean(task.obstacles)),
                    "Median (ms)": 1000 * np.median(samples), "Spread (ms)": 1000 * (max(samples) - min(samples))})
timings = pd.DataFrame(timings)
percentiles = pd.DataFrame([{"Statistic": name, "Milliseconds": timings["Median (ms)"].quantile(q)}
                            for name, q in (("p50", 0.5), ("p90", 0.9), ("p95", 0.95), ("max", 1.0))])
show(percentiles, f"Table 2. Warm request time over {len(timings)} tasks ({repeats} repetitions each).", "{:,.2f}")

fig, (ax_hist, ax_len) = plt.subplots(1, 2, figsize=(11, 3.7))
fig.subplots_adjust(top=0.78, wspace=0.25)
ax_hist.hist(timings["Median (ms)"], bins=20, color="#238b45", alpha=0.85, edgecolor="white", linewidth=0.6)
for name, value in zip(percentiles.Statistic[:2], percentiles.Milliseconds[:2]):
    ax_hist.axvline(value, color=INK, linewidth=1, linestyle=":")
    ax_hist.annotate(name, (value, ax_hist.get_ylim()[1] * 0.92), xytext=(3, 0), textcoords="offset points", fontsize=8, color=INK)
ax_hist.set(xlabel="Warm request time (ms)", ylabel="Tasks", title="Distribution")
ax_len.scatter(timings["Route cells"], timings["Median (ms)"], s=18, color="#238b45", alpha=0.8, linewidths=0)
ax_len.set(xlabel="Route length (cells)", ylabel="Warm request time (ms)", title="Time against route length")
ax_len.grid(True, axis="both")
figure_title(fig, "Warm request time of the canonical A* planner",
             f"{len(timings)} {split} tasks · {resolution}×{resolution} · {OBJECTIVE_LABELS[objective]} objective · {condition}")
figures = save_figure(fig, output_path, f"astar_inference_timing-{resolution}")
plt.show()

*Figure 1. Left: warm request time per task, with the median (p50) and 90th percentile (p90) marked.
Right: time against the length of the returned route. Longer routes usually need more expansions.*

## Summary

In [ ]:
display(Markdown("\n".join([
    f"- The first call, which compiles the search or loads it from cache, takes a one-off {first_call:.2f} s.",
    f"- Warm requests take {percentiles.Milliseconds[0]:.1f} ms at the median and {percentiles.Milliseconds[2]:.1f} ms "
    f"at the 95th percentile over {len(timings)} tasks on this machine.",
    "- A fair comparison times every planner on the same tasks, on the same idle machine, with warm-up "
    "declared in advance and the whole procedure inside the timer, decoding and fallbacks included.",
])))
report = {"resolution": resolution, "split": split, "objective": objective, "condition": condition,
          "machine": machine.to_dict("records"), "first_call_s": first_call,
          "percentiles_ms": percentiles.to_dict("records"), "per_task": timings.to_dict("records"),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"astar_inference_timing-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*